concatinate long format of different data sources to one combined df
manually align the df structure from different sources 

In [1]:
import re
from dateutil import parser

def extract_birth_year_from_dob(dob):
    """
    Extracts the birth year from a date of birth string, handling multiple formats.
    
    Parameters
    ----------
    dob : str or int
        Date of birth in various possible formats (e.g., 'YYYY-MM-DD', 'MM/DD/YYYY',
        'YYYYMMDD', etc.).
        
    Returns
    -------
    int or None
        The birth year as an integer, or None if extraction fails.
    """
    # First, convert dob to string if it's not already
    dob_str = str(dob)
    try:
        # Use dateutil.parser to parse the date string regardless of format.
        parsed_date = parser.parse(dob_str)
        
        # Now, format the parsed date as 'YYYYMMDD'
        formatted_date_str = parsed_date.strftime('%Y%m%d')
        
        # Extract the year from the formatted string and return as an integer.
        return int(formatted_date_str[:4])
    except (ValueError, TypeError, parser.ParserError):
        # Return None if parsing or conversion fails.
        return None

In [2]:
import statsmodels.formula.api as smf
import pandas as pd          
import matplotlib.pyplot as plt 
import numpy as np
from tqdm import tqdm
import seaborn as sns

In [3]:
# HardDrive_thirties
HardDrive_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_thirties_long_format.csv")
HardDrive_thirties_long_format['source'] = 'HardDrive_thirties'

HardDrive_over_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_over_thirties_long_format.csv")
HardDrive_over_thirties_long_format['source'] = 'HardDrive_over_thirties'

HardDrive_under_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_under_thirties_long_format.csv")
HardDrive_under_thirties_long_format['source'] = 'HardDrive_under_thirties'

# add AllTheRages 30to35 to combined_df
drive_30to35 = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/drive_30to35_long_format.csv")
drive_30to35['source'] = 'drive_30to35'

snbb = pd.read_pickle('/home/gaia/Projects/Gaia_gm_vol_cleaned_with_unique_id.pkl').reset_index(drop=False)
snbb['source'] = 'snbb'
snbb.shape
print(snbb.columns)


print(f"amount of unique region labels: {snbb['index'].nunique()}")
snbb_unique_indexes = snbb['index'].unique()

# make index column int and add 1 to each value
snbb['index'] = snbb['index'].astype(int) + 1
sample = snbb.head(1000)



/tmp/ipykernel_168556/329353132.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  HardDrive_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_thirties_long_format.csv")
/tmp/ipykernel_168556/329353132.py:5: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  HardDrive_over_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_over_thirties_long_format.csv")
/tmp/ipykernel_168556/329353132.py:8: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  HardDrive_under_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_under_thirties_long_format.csv")


Index(['level_0', 'index', 'birth_year', 'dob', 'age_at_scan', 'sex',
       'session_id', 'protocol_name', 'name', 'base_name', 'Label Name',
       'network', 'component', 'hemisphere', 'volume', 'metric', 'tissue',
       'tiv', 'total_gm_volume', 'unique_id', 'source'],
      dtype='object')
amount of unique region labels: 454


In [4]:
# fill every cell in snbb tissue column that is missing with 'gm_volume_mm3'
snbb['tissue'] = snbb['tissue'].fillna('gm_volume_mm3')
# print amount of rows with tissue is missing 
print("Rows with missing tissue SNBB:", snbb['tissue'].isnull().sum())

snbb_sample = snbb.head(1000)

Rows with missing tissue SNBB: 0


Aling GNBB

In [5]:
gnbb_volumes = pd.read_csv('/home/gaia/Projects/gnbb.csv')

gnbb_volumes['source'] = 'gnbb_CDs'

gnbb_volumes['session_id'] = gnbb_volumes['session_id'].astype(str)

# remove 'ses-' from session_id column in gnbb_volumes
gnbb_volumes['session_id'] = gnbb_volumes['session_id'].str.replace('ses-', '')

# remove 'sub-' from subject_id column in gnbb_volumes
gnbb_volumes['subject_id'] = gnbb_volumes['subject_id'].str.replace('sub-', '')

# Drop duplicated key columns after merge
gnbb_volumes.drop(columns=['unique_id', 'scan_date'], inplace=True, errors='ignore')

gnbb_volumes['birth_year'] = gnbb_volumes['dob'].apply(extract_birth_year_from_dob)



Align region label to be only the index

In [6]:
print("GNBB Volumes DataFrame Columns:")
print(gnbb_volumes.columns)

print("SNBB column names:")
print(snbb.columns)

# remove the string "schaefer2018tian2020_400_7_" from the region_label column in gnbb_volumes
gnbb_volumes['region_label'] = gnbb_volumes['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

drive_30to35['region_label'] = drive_30to35['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

HardDrive_thirties_long_format['region_label'] = HardDrive_thirties_long_format['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

# in the refactored version (of over and under thirties) the region label is a number, without the whole string
# HardDrive_over_thirties_long_format['region_label'] = HardDrive_over_thirties_long_format['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

# HardDrive_under_thirties_long_format['region_label'] = HardDrive_under_thirties_long_format['region_label'].str.replace('schaefer2018tian2020_400_7_', '')


GNBB Volumes DataFrame Columns:
Index(['subject_id', 'session_id', 'atlas_name', 'region_label', 'tissue_type',
       'volume_mm3', 'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years',
       'dob', 'csf_volume_cm3', 'gm_volume_cm3', 'wm_volume_cm3', 'protocol',
       'source', 'birth_year'],
      dtype='object')
SNBB column names:
Index(['level_0', 'index', 'birth_year', 'dob', 'age_at_scan', 'sex',
       'session_id', 'protocol_name', 'name', 'base_name', 'Label Name',
       'network', 'component', 'hemisphere', 'volume', 'metric', 'tissue',
       'tiv', 'total_gm_volume', 'unique_id', 'source'],
      dtype='object')


Align SNBB

In [7]:
# drop the following columns from snbb: name, base_name, Label Name, network, component, hemisphere, metric
snbb = snbb.drop(columns=['name', 'base_name', 'Label Name', 'network', 'component', 'hemisphere', 'metric'], errors='ignore')

# drop the following clumns from gnbb_volumes: atlas_name, 'csf_volume_cm3', 'wm_volume_cm3'
gnbb_volumes = gnbb_volumes.drop(columns=['atlas_name', 'csf_volume_cm3', 'wm_volume_cm3'], errors='ignore')

# Define SNBB → GNBB column mapping
snbb_to_gnbb = {
    "unique_id": "subject_id",
    "session_id": "session_id",
    "protocol_name": "protocol",
    "sex": "sex",
    "dob": "dob",
    "birth_year": "birth_year",
    "volume": "volume_mm3",
    "tissue": "tissue_type",
    "age_at_scan": "age_in_years",
    "index": "region_label", 
    "tiv": "tiv",
    "total_gm_volume" : "gm_volume_cm3", 
    "source": "source"
}

# Rename SNBB columns
snbb = snbb.rename(columns=snbb_to_gnbb)

# Get all columns from GNBB
gnbb_cols = gnbb_volumes.columns

# Align SNBB columns to GNBB schema (add missing ones as NaN)
snbb = snbb.reindex(columns=gnbb_cols)
print(f"snbb columns are: {snbb.columns}")
snbb_head = snbb.head(100)

snbb.describe()



snbb columns are: Index(['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3',
       'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob',
       'gm_volume_cm3', 'protocol', 'source', 'birth_year'],
      dtype='object')


,region_label,volume_mm3,tiv,institute,manufacturer,gm_volume_cm3,birth_year
count,1.797386e+06,1.797386e+06,1.797386e+06,0.0,0.0,1.797386e+06,1.797386e+06
mean,2.275000e+02,1.209263e+03,1.428730e+03,NaN,NaN,5.490055e+05,1.991786e+03
std,1.310582e+02,6.351395e+02,1.345469e+02,NaN,NaN,5.340490e+04,9.956250e+00
min,1.000000e+00,7.122278e-01,1.031780e+03,NaN,NaN,3.471330e+05,1.931000e+03
25%,1.140000e+02,7.496608e+02,1.334070e+03,NaN,NaN,5.114435e+05,1.988000e+03
50%,2.275000e+02,1.104961e+03,1.431330e+03,NaN,NaN,5.492299e+05,1.994000e+03
75%,3.410000e+02,1.547557e+03,1.520690e+03,NaN,NaN,5.854327e+05,1.999000e+03
max,4.540000e+02,5.844354e+03,1.908020e+03,NaN,NaN,7.139129e+05,2.015000e+03


### fill institute and manufacturer for snbb 
Scanner link - https://mri.tau.ac.il/Human_3T_Prisma

In [8]:
# is there any value that is not null in the institute column of snbb
not_missing_institues = snbb[snbb['institute'].notna()]['institute'].unique()
print(f"Institutes not missing in SNBB: {not_missing_institues}")

# fill institute with 'Tel-Aviv University' 
snbb['institute'] = snbb['institute'].fillna('Tel-Aviv University')

not_missing_manufacturers = snbb[snbb['manufacturer'].notna()]['manufacturer'].unique()
print(f"Manufacturers not missing in SNBB: {not_missing_manufacturers}")

# fill manufacturer with 'SIEMENS'
snbb['manufacturer'] = snbb['manufacturer'].fillna('SIEMENS')

# validate no missing values in institute and manufacturer columns
print("Rows with missing institute SNBB:", snbb['institute'].isnull().sum())
print("Rows with missing manufacturer SNBB:", snbb['manufacturer'].isnull().sum())

Institutes not missing in SNBB: []
Manufacturers not missing in SNBB: []
Rows with missing institute SNBB: 0
Rows with missing manufacturer SNBB: 0


In [9]:
# Concatenate
combined_df = pd.concat([gnbb_volumes, snbb, drive_30to35, HardDrive_thirties_long_format, HardDrive_over_thirties_long_format, HardDrive_under_thirties_long_format], ignore_index=True)

print("Combined shape:", combined_df.shape)
print("Columns:", combined_df.columns.tolist())
print(f"amount of unique region labels: {combined_df['region_label'].nunique()}")
print(f"amount of unique subject ids: {combined_df['subject_id'].nunique()}")

#save a list of unique region labels
unique_region_labels = combined_df['region_label'].unique()
#types of values in unique_region_labels
print(f"types of values in unique_region_labels: {type(unique_region_labels[0])}")

# change all region labels to int
combined_df['region_label'] = combined_df['region_label'].astype(int)
print(f"amount of unique region labels: {combined_df['region_label'].nunique()}")

print(f"head of combined is: {combined_df.head(10)}")



Combined shape: (3219455, 24)
Columns: ['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3', 'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob', 'gm_volume_cm3', 'protocol', 'source', 'birth_year', 'Unnamed: 0', 'atlas_name', 'scan_time', 'age_at_scan', 'weight', 'directory_path', 'estimated_critical_info', 'scan_date', 'file_path']
amount of unique region labels: 908
amount of unique subject ids: 3748
types of values in unique_region_labels: <class 'str'>
amount of unique region labels: 454
head of combined is:   subject_id session_id  region_label    tissue_type   volume_mm3     tiv sex  \
0      ls191   20070322             1  gm_volume_mm3   907.389714  1257.9   M   
1      ls191   20070322             2  gm_volume_mm3  1493.151433  1257.9   M   
2      ls191   20070322             3  gm_volume_mm3  1097.402352  1257.9   M   
3      ls191   20070322             4  gm_volume_mm3  1316.549873  1257.9   M   
4      ls191   20070322             5  gm_v

In [10]:
# keep only rows where tissue_type is 'gm_volume_mm3'
combined_df = combined_df[combined_df['tissue_type'] == 'gm_volume_mm3']

print(f"Rows with missing tissue_type combined: {combined_df['tissue_type'].isnull().sum()}")

# apply extract_birth_year_from_dob to the dob column to create a new birth_year column
combined_df['birth_year'] = combined_df['dob'].apply(extract_birth_year_from_dob)

# # remove duplications based on subject_id , session_id, region_label, tissue_type
# combined_df = combined_df.drop_duplicates(subset=['subject_id', 'session_id', 'region_label', 'tissue_type'])


Rows with missing tissue_type combined: 0


In [11]:
# rows with missing institute
missing_institute_rows = combined_df[combined_df['institute'].isnull()]
print(f"Rows with missing institute in combined_df: {missing_institute_rows.shape[0]}")

# unique institutes in combined_df
unique_institutes = combined_df['institute'].unique()
print(f"Unique institutes in combined_df: {unique_institutes}")

# replace 'Tel Aviv Medical Center' and 'TEL AVIV MEDICAL CENTER' with 'ICHILOV TEL AVIV'
combined_df['institute'] = combined_df['institute'].replace(['Tel Aviv Medical Center', 'TEL AVIV MEDICAL CENTER'], 'ICHILOV TEL AVIV')

Rows with missing institute in combined_df: 0
Unique institutes in combined_df: ['Tel Aviv Medical Center' 'TEL AVIV MEDICAL CENTER' 'ICHILOV TEL AVIV'
 'SHEBA_ MEDICAL_ CENTER' 'Tel-Aviv University']


In [12]:
# remove "ses-"
combined_df['session_id'] = combined_df['session_id'].str.replace('ses-', '')

# remove "sub-"
combined_df['subject_id'] = combined_df['subject_id'].str.replace('sub-', '')

In [13]:
# fill scan_date according to session_id (yyyymmdd) to date format
combined_df['session_id'] = combined_df['session_id'].astype(str)

combined_df['scan_date'] = combined_df['session_id'].apply(
    lambda x: f"{x[:4]}-{x[4:6]}-{x[6:8]}" if len(str(x)) in [8, 12] else None
)
combined_df['scan_year'] = combined_df['scan_date'].apply(lambda x: x.split('-')[0] if x else None)

combined_df['scan_year'] = pd.to_numeric(combined_df['scan_year'], errors='coerce')
 

# tests
to make sure all important info is there

In [14]:
# rows without volume_mm3
rows_without_volume = combined_df[combined_df['volume_mm3'].isnull()]
print(f"Rows without volume_mm3: {rows_without_volume.shape[0]}")

# rows without 'sex', 'age_in_years', 'dob', 'birth_year'
rows_missing_important_info = combined_df[
    combined_df['sex'].isnull() |
    combined_df['age_in_years'].isnull() |
    combined_df['dob'].isnull() |
    combined_df['birth_year'].isnull() | 
    combined_df['tiv'].isnull() | 
    combined_df['session_id'].isnull()

]
print(f"Rows missing important info: {rows_missing_important_info.shape[0]}")

# remove rows with missing important info from combined_df
combined_df = combined_df.drop(rows_missing_important_info.index)

# make sure it worked 
rows_missing_important_info_after = combined_df[
    combined_df['sex'].isnull() |
    combined_df['age_in_years'].isnull() |
    combined_df['dob'].isnull() |
    combined_df['birth_year'].isnull() | 
    combined_df['tiv'].isnull() | 
    combined_df['session_id'].isnull()
]
print(f"Rows missing important info after removal: {rows_missing_important_info_after.shape[0]}")

Rows without volume_mm3: 0
Rows missing important info: 8172
Rows missing important info after removal: 0


In [15]:
print(f"columns with missing values: {rows_missing_important_info.columns[rows_missing_important_info.isnull().any()].tolist()}")

print(f"sources of rows without tiv: {rows_missing_important_info[rows_missing_important_info['tiv'].isna()]['source'].unique()}")

columns with missing values: ['tiv', 'sex', 'gm_volume_cm3', 'Unnamed: 0', 'directory_path', 'scan_date', 'scan_year']
sources of rows without tiv: ['HardDrive_over_thirties']


In [16]:
# save combined_df to a pickle file
combined_df.to_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')